# Inspect mesh-based crypt segmentations

Single-notebook workflow to **browse and visualize crypt segmentations produced by the new mesh-based pipeline**.

It:
- scans saved segmentation files (`.npz`)
- builds a small inspection table with per-organoid summary information
- loads the corresponding organoid mesh for a selected segmentation
- plots the mesh with detected crypt regions highlighted
- overlays the detected crypt bottoms as points



In [ ]:

# If you edit paths, re-run this cell and everything below.
import os, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    HAS_WIDGETS = True
except Exception:
    from IPython.display import display, HTML
    HAS_WIDGETS = False

from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.plotting.meshes import plot_mesh_by_regions, add_mesh_overlay

# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Folder containing saved mesh-based segmentation outputs
# SEG_DIR = os.path.join(PROJECT_ROOT, "..", "NicoleData", "20251201", "crypt_segmentations_mesh")
# SEG_DIR = os.path.join(PROJECT_ROOT, "..", "NicoleData", "20250929", "crypt_segmentations_mesh_day3p5_prct4")
SEG_DIR = os.path.join(PROJECT_ROOT, "..", "NicoleData", "perturbations_day3p5", "crypt_segmentations_mesh")

# Which timepoints to inspect. Set to None to auto-discover from SEG_DIR.
TIMEPOINTS = ['day3p5'] # None

# Plot settings
BACKEND = "plotly"
BASELINE_COLOR = "lightgray"
BOTTOM_SIZE = 10.0

print("SEG_DIR =", SEG_DIR)


SEG_DIR = /home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData/20250929/crypt_segmentations_mesh_day3p5_prct4


## Utilities

In [2]:

def discover_timepoints(seg_dir):
    if not os.path.isdir(seg_dir):
        return []
    tps = []
    for name in sorted(os.listdir(seg_dir)):
        p = os.path.join(seg_dir, name)
        if os.path.isdir(p) and glob.glob(os.path.join(p, "*.npz")):
            tps.append(name)
    return tps


def _npz_scalar(x):
    arr = np.asarray(x)
    if arr.shape == ():
        try:
            return arr.item()
        except Exception:
            return arr
    return arr


def _patches_from_ll(crypts_ll):
    """Convert saved list-of-lists / object array into list[set[int]]."""
    if crypts_ll is None:
        return []

    arr = np.asarray(crypts_ll, dtype=object)

    # Empty
    if arr.size == 0:
        return []

    out = []
    for p in arr.tolist():
        if p is None:
            out.append(set())
        else:
            out.append(set(int(v) for v in p))
    return out


def build_inspection_table(seg_dir, timepoints=None, verbose=True, max_items=None):
    if timepoints is None:
        timepoints = discover_timepoints(seg_dir)

    rows = []
    n_seen = 0

    for tp in timepoints:
        tp_dir = os.path.join(seg_dir, tp)
        for seg_path in sorted(glob.glob(os.path.join(tp_dir, "*.npz"))):
            if max_items is not None and n_seen >= int(max_items):
                break

            try:
                z = np.load(seg_path, allow_pickle=True)
            except Exception as e:
                if verbose:
                    print(f"[skip] could not load {seg_path}: {e}")
                continue

            label_uid = str(_npz_scalar(z["label_uid"])) if "label_uid" in z else os.path.splitext(os.path.basename(seg_path))[0]
            mesh_path = str(_npz_scalar(z["mesh_path"])) if "mesh_path" in z else None
            crypts_ll = z["crypts_ll"] if "crypts_ll" in z else np.array([], dtype=object)
            bottom_ids = z["bottom_vertex_ids"] if "bottom_vertex_ids" in z else np.array([], dtype=np.int64)

            n_crypts = len(np.asarray(crypts_ll, dtype=object))
            rows.append(dict(
                timepoint=tp,
                label_uid=label_uid,
                seg_path=seg_path,
                mesh_path=mesh_path,
                n_crypts=int(n_crypts),
                n_bottoms=int(np.asarray(bottom_ids).size),
                has_seg=True,
            ))
            n_seen += 1

    return pd.DataFrame(rows)


def filter_table(df, *, timepoint=None, n_crypts_min=None, n_crypts_max=None, require_seg=True):
    out = df.copy()
    if timepoint is not None:
        out = out[out["timepoint"] == timepoint]
    if require_seg:
        out = out[out["has_seg"] == True]
    if n_crypts_min is not None:
        out = out[out["n_crypts"] >= int(n_crypts_min)]
    if n_crypts_max is not None:
        out = out[out["n_crypts"] <= int(n_crypts_max)]
    return out.reset_index(drop=True)


def plot_one_organoid_mesh_seg(
    seg_path,
    *,
    backend="plotly",
    baseline_color="lightgray",
    bottom_size=10.0,
):
    """
    Load one saved mesh-based segmentation and visualize:
      - mesh with crypt regions highlighted
      - crypt bottoms as overlaid points
    """
    z = np.load(seg_path, allow_pickle=True)

    if "mesh_path" not in z:
        raise KeyError(f"{seg_path} does not contain 'mesh_path'.")

    mesh_path = str(_npz_scalar(z["mesh_path"]))
    crypts_ll = z["crypts_ll"] if "crypts_ll" in z else np.array([], dtype=object)
    bottom_vertex_ids = z["bottom_vertex_ids"] if "bottom_vertex_ids" in z else np.array([], dtype=np.int64)

    crypts = _patches_from_ll(crypts_ll)
    bottom_vertex_ids = np.asarray(bottom_vertex_ids, dtype=np.int64)

    mesh = OrganoidMesh(str(mesh_path))
    mesh.normalize_inplace()

    fig = plot_mesh_by_regions(
        mesh,
        crypts,
        backend=backend,
        region_kind="auto",
        region_names=[f"crypt {i+1}" for i in range(len(crypts))],
        baseline_color=baseline_color,
    )

    if bottom_vertex_ids.size > 0:
        add_mesh_overlay(
            fig,
            mesh,
            bottom_vertex_ids,
            backend=backend,
            mode="points",
            size=float(bottom_size),
            name="crypt bottoms",
        )

    return fig, mesh, crypts, bottom_vertex_ids


## Build inspection table

In [3]:

df_inspect = build_inspection_table(SEG_DIR, timepoints=TIMEPOINTS, verbose=True)
print(f"Found {len(df_inspect)} saved segmentations")
df_inspect.head()


Found 312 saved segmentations


,timepoint,label_uid,seg_path,mesh_path,n_crypts,n_bottoms,has_seg
0,day3p5,day3p5_A01_1,/home/fmoller/Projects/LearningOrganoids/Organ...,/home/fmoller/Projects/LearningOrganoids/Nicol...,1,1,True
1,day3p5,day3p5_A01_10,/home/fmoller/Projects/LearningOrganoids/Organ...,/home/fmoller/Projects/LearningOrganoids/Nicol...,0,0,True
2,day3p5,day3p5_A01_11,/home/fmoller/Projects/LearningOrganoids/Organ...,/home/fmoller/Projects/LearningOrganoids/Nicol...,0,0,True
3,day3p5,day3p5_A01_12,/home/fmoller/Projects/LearningOrganoids/Organ...,/home/fmoller/Projects/LearningOrganoids/Nicol...,2,2,True
4,day3p5,day3p5_A01_13,/home/fmoller/Projects/LearningOrganoids/Organ...,/home/fmoller/Projects/LearningOrganoids/Nicol...,1,1,True


In [4]:
# Stacked bar chart: number of organoids with 0, 1, 2, ... crypts at each timepoint

if len(df_inspect) == 0:
    print("No segmentations found.")
else:
    counts = (
        df_inspect
        .groupby(["timepoint", "n_crypts"])
        .size()
        .unstack(fill_value=0)
        .sort_index(axis=1)
    )

    fig_counts = go.Figure()

    for n_crypts in counts.columns:
        fig_counts.add_bar(
            x=counts.index.tolist(),
            y=counts[n_crypts].tolist(),
            name=f"{int(n_crypts)} crypts",
        )

    fig_counts.update_layout(
        barmode="stack",
        title="Number of organoids by timepoint and crypt count",
        xaxis_title="Timepoint",
        yaxis_title="Number of organoids",
        legend_title="Crypt count",
    )

    fig_counts.show()

## Quick start (no widgets)

In [5]:

if len(df_inspect) == 0:
    print("No segmentations found. Check SEG_DIR and TIMEPOINTS.")
else:
    tp0 = df_inspect["timepoint"].iloc[0]
    df_f = filter_table(df_inspect, timepoint=tp0, n_crypts_min=1, require_seg=True)
    df_f.head()


In [6]:

n_show = min(3, len(df_f)) if "df_f" in globals() else 0

for i in range(n_show):
    r = df_f.iloc[i]
    display(HTML(f"<h4>{i+1}/{n_show}: {r['label_uid']} (crypts={int(r['n_crypts'])})</h4>"))

    fig, mesh, crypts, bottom_vertex_ids = plot_one_organoid_mesh_seg(
        r["seg_path"],
        backend=BACKEND,
        baseline_color=BASELINE_COLOR,
        bottom_size=BOTTOM_SIZE,
    )
    display(fig)


## Optional widget browser

In [7]:
if HAS_WIDGETS and len(df_inspect) > 0:
    tps = sorted(df_inspect["timepoint"].unique().tolist())
    max_crypts = int(df_inspect["n_crypts"].max())

    tp_dd = widgets.Dropdown(
        options=tps,
        value=(tps[0] if tps else None),
        description="Timepoint:"
    )

    nmin = widgets.IntSlider(
        value=0,
        min=0,
        max=max_crypts,
        step=1,
        description="Crypts ≥"
    )

    nmax = widgets.IntSlider(
        value=max_crypts,
        min=0,
        max=max_crypts,
        step=1,
        description="Crypts ≤"
    )

    nshow = widgets.IntSlider(
        value=3,
        min=1,
        max=12,
        step=1,
        description="Show"
    )

    shuffle = widgets.Checkbox(
        value=False,
        description="Shuffle"
    )

    out = widgets.Output()

    def _refresh_table():
        df_f = filter_table(
            df_inspect,
            timepoint=tp_dd.value,
            n_crypts_min=nmin.value,
            n_crypts_max=nmax.value,
            require_seg=True,
        ).reset_index(drop=True)

        return df_f

    def _draw(*args):
        df_f = _refresh_table()

        with out:
            clear_output(wait=True)

            if len(df_f) == 0:
                print("No organoids match the current filters.")
                return

            if shuffle.value:
                df_plot = df_f.sample(frac=1).reset_index(drop=True)
            else:
                df_plot = df_f.reset_index(drop=True)

            n_to_show = min(nshow.value, len(df_plot))

            display(HTML(
                f"<h4>{len(df_f)} matching organoids "
                f"(showing {n_to_show})</h4>"
            ))

            for i in range(n_to_show):
                r = df_plot.iloc[i]

                display(HTML(
                    f"<h4>{i+1}/{n_to_show}: {r['label_uid']} "
                    f"(timepoint={r['timepoint']}, crypts={int(r['n_crypts'])})</h4>"
                ))

                fig, mesh, crypts, bottom_vertex_ids = plot_one_organoid_mesh_seg(
                    r["seg_path"],
                    backend=BACKEND,
                    baseline_color=BASELINE_COLOR,
                    bottom_size=BOTTOM_SIZE,
                )
                display(fig)

    tp_dd.observe(_draw, names="value")
    nmin.observe(_draw, names="value")
    nmax.observe(_draw, names="value")
    nshow.observe(_draw, names="value")
    shuffle.observe(_draw, names="value")

    controls = widgets.VBox([tp_dd, nmin, nmax, nshow, shuffle])
    display(controls, out)
    _draw()

elif not HAS_WIDGETS:
    print("ipywidgets not available in this environment; use the quick-start cells above.")
else:
    print("No segmentations found.")

Output()